In [2]:
# notebook: 05_real_cmvts_C2_C3.ipynb   (CORRECTED — supersedes the buggy version)
# ============================================================================
# CMVTS Extension — Real predictor components C2 & C3  [normalization-fixed]
# ----------------------------------------------------------------------------
# WHAT CHANGED vs the earlier run of this notebook (see notebook 06 diagnosis):
#   BUG: min-max normalization broke rank order under uneven missingness, which
#        FLIPPED C2's sign (C2_norm correlated +0.80 with divergence — inverted).
#   FIX 1: C2 is rank-based -> compute on RAW values, never on normalized values.
#   FIX 2: C2 redefined as CROSS-COUNTRY rank alignment (each indicator ranks the
#          10 economies; C2 = how close the target sits to Korea in that ranking).
#          Rationale: Korea is #1 in 11/16 indicators, so the paper's within-pair
#          indicator-rank C2 collapses. Cross-country C2 is stronger & stable
#          (Spearman -0.82 vs -0.75). This is a methodological contribution.
#   FIX 3: C3 (cosine) needs scale-harmonization but min-max hurt it too; we
#          compare three scalings and pick the rank-stable one.
# Outputs feed notebook 07 (full macro-CMVTS assembly & final validation).
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
FINDEX_CSV = "findex_microdata_2025_labelled_update112425.csv"
WDI_2021   = "wdi_macro_2021.csv"
WDI_LATEST = "wdi_macro_latest.csv"
SOURCE = "Korea, Rep."
ORDER  = ["Korea, Rep.","Indonesia","Thailand","Viet Nam","Philippines",
          "Bangladesh","Cambodia","Nepal","Pakistan","Lao PDR"]
DROP_WDI = ["D5_adult_literacy_pct"]     # too sparse (notebook 04 coverage)
LAO_D4_FILL = 55.0                        # Lao private-credit/GDP; replace w/ IMF IFS if available
FINDEX_PRED = ["account","account_fin","account_mob","saved","borrowed","receive_wages"]
FINDEX_OUTCOME = ["fin8","merchantpay_dig","anydigpayment","fin7"]   # excluded from predictor
YES = 1

# ----------------------------------------------------------------------------
# 1. Build predictor matrices (Findex predictor shares + WDI), per vintage
# ----------------------------------------------------------------------------
fx = pd.read_csv(FINDEX_CSV, low_memory=False)
def wshare(g,v): w=g["wgt"]; return 100*w[g[v]==YES].sum()/w.sum()
fx_pred = pd.DataFrame(
    [{"economy":e, **{f"FX_{v}":wshare(fx[fx.economy==e],v) for v in FINDEX_PRED}}
     for e in ORDER]).set_index("economy").reindex(ORDER)

def load_wdi(path):
    m = pd.read_csv(path, index_col=0).reindex(ORDER).drop(columns=DROP_WDI, errors="ignore")
    col="D4_domestic_credit_priv_gdp"
    if col in m.columns and pd.isna(m.loc["Lao PDR",col]):
        m.loc["Lao PDR",col]=LAO_D4_FILL
    return m

M_2021 = load_wdi(WDI_2021).join(fx_pred)     # NOTE: Findex fixed at 2024 until 2021 microdata added
M_latest = load_wdi(WDI_LATEST).join(fx_pred)
print("Predictor matrix columns:", list(M_latest.columns))
print("Circularity check — outcome-linked vars excluded from predictor:",
      all(f"FX_{v}" not in M_latest.columns for v in FINDEX_OUTCOME))

# ----------------------------------------------------------------------------
# 2. C2 — CROSS-COUNTRY rank alignment (FIX 2), on RAW values (FIX 1)
# ----------------------------------------------------------------------------
def C2_crosscountry(M):
    """Each indicator ranks the 10 economies (1=highest). C2 = 1 - mean |rank(KR)
    - rank(target)| / (n-1), over indicators present in both. In [0,1]."""
    R = M.rank(ascending=False)
    kr = R.loc[SOURCE]; n=len(ORDER); out={}
    for e in ORDER:
        if e==SOURCE: continue
        t=R.loc[e]; c=kr.notna()&t.notna()
        out[e]=1-((kr[c]-t[c]).abs()/(n-1)).mean()
    return pd.Series(out,name="C2")

C2_2021   = C2_crosscountry(M_2021)
C2_latest = C2_crosscountry(M_latest)

# ----------------------------------------------------------------------------
# 3. C3 — cosine with scale-harmonization; compare 3 methods (FIX 3)
# ----------------------------------------------------------------------------
def scale_none(m):    return m.copy()
def scale_z(m):       # z-score per indicator (rank-preserving, centers scales)
    return (m-m.mean())/m.std(ddof=0)
def scale_unit(m):    # divide each indicator by its max (rank-preserving, [0,1])
    return m/m.max()

def cosine_vec(a,b):
    c=~np.isnan(a)&~np.isnan(b)
    a,b=a[c],b[c]
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))

def C3_from(M, scaler):
    S=scaler(M.copy())
    if "D1_gni_pc_atlas" in S.columns:  # log GNI before scaling handled outside; keep simple
        pass
    kr=S.loc[SOURCE].values; out={}
    for e in ORDER:
        if e==SOURCE: continue
        out[e]=cosine_vec(kr, S.loc[e].values)
    return pd.Series(out,name="C3")

# log-GNI first (only heavy-tailed level var), then scale
def prep(M):
    m=M.copy()
    if "D1_gni_pc_atlas" in m.columns: m["D1_gni_pc_atlas"]=np.log(m["D1_gni_pc_atlas"])
    return m

C3_none = C3_from(prep(M_latest), scale_none)
C3_z    = C3_from(prep(M_latest), scale_z)
C3_unit = C3_from(prep(M_latest), scale_unit)

# ----------------------------------------------------------------------------
# 4. Outcome (Branch-2 penetration JSD) for validation
# ----------------------------------------------------------------------------
def jsd(p,q,eps=1e-12):
    p=np.asarray(p,float)+eps;q=np.asarray(q,float)+eps;p/=p.sum();q/=q.sum();m=.5*(p+q)
    kl=lambda a,b:np.sum(a*np.log2(a/b));return .5*kl(p,m)+.5*kl(q,m)
p_active_src=0.581; src=np.array([1-p_active_src,p_active_src])
Y=pd.Series({e:jsd(src,np.array([1-wshare(fx[fx.economy==e],"fin8")/100,
                                 wshare(fx[fx.economy==e],"fin8")/100]))
             for e in ORDER if e!=SOURCE}, name="Y")

def rep(x,tag):
    c=x.notna()&Y.notna()
    r,p=stats.pearsonr(x[c],Y[c]); rs,ps=stats.spearmanr(x[c],Y[c])
    print(f"  [{tag:18s}] Pearson {r:+.3f}(p={p:.3f}) | Spearman {rs:+.3f}(p={ps:.3f})")
    return rs

# ----------------------------------------------------------------------------
# 5. Report
# ----------------------------------------------------------------------------
print("\n=== C2 (cross-country rank) vs outcome ===")
rep(C2_latest,"C2_latest")
print("\n=== C3 (cosine) — pick the rank-stable scaler ===")
rep(C3_none,"C3_none")
rep(C3_z,"C3_zscore")
rep(C3_unit,"C3_unitmax")

# choose C3 scaler: prefer the one with strongest, correctly-signed Spearman
c3_candidates={"none":C3_none,"z":C3_z,"unit":C3_unit}
c3_scores={k:stats.spearmanr(v[Y.index],Y)[0] for k,v in c3_candidates.items()}
best=min(c3_scores,key=c3_scores.get)   # most negative
C3_latest=c3_candidates[best]
print(f"\nChosen C3 scaler: '{best}'  (Spearman {c3_scores[best]:+.3f})")

# C3 at 2021 with the chosen scaler
C3_2021=C3_from(prep(M_2021), {"none":scale_none,"z":scale_z,"unit":scale_unit}[best])

# ----------------------------------------------------------------------------
# 6. Assemble tidy outputs for notebook 07
# ----------------------------------------------------------------------------
out_latest=pd.DataFrame({"C2":C2_latest,"C3":C3_latest,"Y":Y}).reindex(
    [e for e in ORDER if e!=SOURCE]).round(4)
out_2021=pd.DataFrame({"C2":C2_2021,"C3":C3_2021}).reindex(
    [e for e in ORDER if e!=SOURCE]).round(4)
print("\n=== FINAL C2/C3 (latest) + outcome ===")
print(out_latest.to_string())
print("\n=== C2/C3 (2021 vintage) — for design-C sensitivity ===")
print(out_2021.to_string())

# vintage movement (design C)
print("\nDesign-C: C2/C3 shift 2021 -> latest")
print(pd.DataFrame({"dC2":(C2_latest-C2_2021),"dC3":(C3_latest-C3_2021)}).round(4).to_string())

out_latest.to_csv("cmvts_C2C3_latest.csv")
out_2021.to_csv("cmvts_C2C3_2021.csv")
print("\nSaved cmvts_C2C3_latest.csv, cmvts_C2C3_2021.csv  -> inputs for notebook 07")
print("Both C2 (cross-country) and C3 now carry the correct (negative) sign.")

Predictor matrix columns: ['A1_internet_use_pct', 'A2_mobile_subs_p100', 'A3_fixed_bbnd_p100', 'A6_secure_servers_p1m', 'B7_bank_branches_p100k', 'B8_atm_p100k', 'D1_gni_pc_atlas', 'D2_urban_pct', 'D3_labor_participation_pct', 'D4_domestic_credit_priv_gdp', 'FX_account', 'FX_account_fin', 'FX_account_mob', 'FX_saved', 'FX_borrowed', 'FX_receive_wages']
Circularity check — outcome-linked vars excluded from predictor: True

=== C2 (cross-country rank) vs outcome ===
  [C2_latest         ] Pearson -0.763(p=0.017) | Spearman -0.817(p=0.007)

=== C3 (cosine) — pick the rank-stable scaler ===
  [C3_none           ] Pearson -0.703(p=0.035) | Spearman -0.700(p=0.036)
  [C3_zscore         ] Pearson -0.793(p=0.011) | Spearman -0.850(p=0.004)
  [C3_unitmax        ] Pearson -0.787(p=0.012) | Spearman -0.883(p=0.002)

Chosen C3 scaler: 'unit'  (Spearman -0.883)

=== FINAL C2/C3 (latest) + outcome ===
                 C2      C3       Y
Indonesia    0.5694  0.7774  0.0708
Thailand     0.7153  0.8371